In [3]:
# Mount Drive (so data survives session restarts)
from google.colab import drive, userdata
drive.mount('/content/drive')

# Pull secrets from Colab Secrets into env vars
import os
for k in ['HF_TOKEN', 'OPENROUTER_API_KEY', 'KAGGLE_USERNAME', 'KAGGLE_KEY']:
    os.environ[k] = userdata.get(k)

# Clone the repo
!git clone https://github.com/jo3591/assigment2dsai413 /content/cxr-intel
%cd /content/cxr-intel
!bash scripts/colab_bootstrap.sh


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/cxr-intel'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 115 (delta 16), reused 114 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 180.22 KiB | 1.50 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/cxr-intel
==> Installing pinned deps
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Cannot install -r requirements-colab.txt (line 6), -r requirements-colab.txt (line 8) and transform

In [4]:
!pip install -q --upgrade "transformers>=4.48,<4.51" "colpali-engine>=0.3.10" \
    "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=1.1" "open_clip_torch>=2.29" \
    "faiss-cpu" "pydicom" "pydantic>=2.9" "python-dotenv" "kaggle" "rapidfuzz" \
    "tenacity" "openai" "bert-score" "sacrebleu" "rouge-score" "sentencepiece" "streamlit"
!pip install -q -e .


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Cannot install colpali-engine==0.3.10, colpali-engine==0.3.11, colpali-engine==0.3.12, colpali-engine==0.3.13, colpali-engine==0.3.14, colpali-engine==0.3.15, colpali-engine==0.3.16 and transformers<4.51 and >=4.48 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.9/

In [6]:
import torch, transformers
from importlib.metadata import version
print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print(f"colpali_engine={version('colpali-engine')}")
print(f"CUDA available: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


torch=2.10.0+cu128
transformers=5.8.1
colpali_engine=0.3.16
CUDA available: True, device: Tesla T4


In [7]:
!python scripts/download_data.py --config configs/data.yaml


20:31:27 | INFO    | cxr_intel.data.kaggle_loader | Downloading kaggle dataset simhadrisadaram/mimic-cxr-dataset -> data/raw
Dataset URL: https://www.kaggle.com/datasets/simhadrisadaram/mimic-cxr-dataset
100% 16.5G/16.5G [03:03<00:00, 96.4MB/s]

20:38:35 | INFO    | download_data | Reports CSV: data/raw/mimic_cxr_aug_train.csv
20:38:39 | INFO    | cxr_intel.data.kaggle_loader | Loaded 64586 rows from data/raw/mimic_cxr_aug_train.csv | columns=['Unnamed: 0.1', 'Unnamed: 0', 'subject_id', 'image', 'view', 'AP', 'PA', 'Lateral', 'text', 'text_augment']
20:38:55 | INFO    | cxr_intel.data.preprocess | Length filter: 64586 -> 57418 (drop reports < 30 tokens)
20:38:55 | INFO    | download_data | Computing rule-based CheXpert labels…
20:40:49 | INFO    | download_data | Stratified subsample to 4000 rows
20:40:49 | INFO    | download_data | Resolving image paths…
20:40:49 | INFO    | download_data | Images located: 0 / 4000
20:40:49 | INFO    | download_data | Splits: train=0, val=0, test=0
20

In [9]:
import os, pathlib
root = pathlib.Path('data/raw/official_data_iccv_final')
print("Top:")
for x in sorted(os.listdir(root))[:15]:
    p = root / x
    print(f"  {'DIR ' if p.is_dir() else 'FILE'} {x}")

# Look for a real CXR
import subprocess
result = subprocess.run(['find', str(root), '-name', '*.jpg', '-not', '-path', '*/staging/*'],
                       capture_output=True, text=True, timeout=20)
sample = result.stdout.split('\n')[:5]
print(f"\nSample images ({len(result.stdout.splitlines())} total):")
for s in sample:
    print(f"  {s}")


Top:
  DIR  files

Sample images (261137 total):
  data/raw/official_data_iccv_final/files/p17/p17251081/s56476430/56579ed7-299ec8c0-1abc3250-6478ff4e-dc15a581.jpg
  data/raw/official_data_iccv_final/files/p17/p17251081/s56476430/2c06022f-7b7c3f10-af7d842a-dbc064fd-d8de86e9.jpg
  data/raw/official_data_iccv_final/files/p17/p17251081/s51960580/7de07c4b-4c555d4f-bab9234e-ba900be5-fb32a15d.jpg
  data/raw/official_data_iccv_final/files/p17/p17251081/s51960580/6a2efec2-a97c7a22-15269dd3-d357893f-809ac3be.jpg
  data/raw/official_data_iccv_final/files/p17/p17251081/s50697534/0a686a30-f8f4a8aa-9d66daf1-c21b5caf-7379e11f.jpg


In [11]:
%cd /content/cxr-intel
!git pull
!python scripts/download_data.py --config configs/data.yaml --skip-download


/content/cxr-intel
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 13 (delta 11), reused 13 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 3.65 KiB | 934.00 KiB/s, done.
From https://github.com/jo3591/assigment2dsai413
   4ec17e4..201462f  main       -> origin/main
Updating 4ec17e4..201462f
Fast-forward
 pyproject.toml                           |  3 +-
 requirements-colab.txt                   | 15 +++++-----
 requirements.txt                         |  3 +-
 scripts/download_data.py                 |  6 +++-
 src/cxr_intel/data/kaggle_loader.py      | 47 +++++++++++++++++++++++++-------
 src/cxr_intel/retrieval/colpali_index.py | 22 +++++++++++++--
 6 files changed, 71 insertions(+), 25 deletions(-)
21:07:57 | INFO    | download_data | Reports CSV: data/raw/mimic_cxr_aug_train.csv
21:07:59 | INFO    | cxr_intel.data.kaggle_loader | Loaded 64586 rows from data/raw/mimi

In [12]:
!mkdir -p /content/drive/MyDrive/cxr_intel_artifacts
!cp -r data/processed /content/drive/MyDrive/cxr_intel_artifacts/
!cp -r data/samples /content/drive/MyDrive/cxr_intel_artifacts/
print("Backed up to Drive.")


Backed up to Drive.


In [13]:
import pandas as pd
df = pd.read_parquet('data/processed/reports.parquet')
print("Columns:", df.columns.tolist())
print("\nLabel distribution:")
print(df['primary_chexpert_label'].value_counts().head(15))
print("\nSample report (truncated):")
print(df.iloc[0]['clean_text'][:300])


Columns: ['Unnamed: 0.1', 'Unnamed: 0', 'subject_id', 'image', 'view', 'AP', 'PA', 'Lateral', 'text', 'text_augment', 'clean_text', 'n_tokens', 'findings', 'impression', 'primary_chexpert_label', 'chexpert_vec', 'image_path', 'study_id']

Label distribution:
primary_chexpert_label
Atelectasis                   1026
Other                          901
Cardiomegaly                   173
Consolidation                  131
Lung Opacity                   129
Edema                          107
No Finding                      90
Lung Lesion                     57
Fracture                        57
Pleural Effusion                54
Support Devices                 52
Pneumonia                       32
Pleural Other                   15
Pneumothorax                     8
Enlarged Cardiomediastinum       3
Name: count, dtype: int64

Sample report (truncated):
['Findings: Impression: ', 'Findings: are grossly unchanged. Interval improvement in vascular congestion is demonstrated, currently with re

In [14]:
!python scripts/build_qa_dataset.py --config configs/data.yaml --limit 20


synth->qa_v1.jsonl:   0% 0/20 [00:00<?, ?it/s]21:23:46 | INFO    | cxr_intel.models.llm_router | LLMRouter using provider=openrouter model=meta-llama/llama-3.3-70b-instruct
21:24:01 | WARNING | cxr_intel.qa_dataset.synth_generator | LLM error on study=s52026137 qtype=existence: RetryError[<Future at 0x7af4e1cbd940 state=finished raised AttributeError>]
21:24:15 | WARNING | cxr_intel.qa_dataset.synth_generator | LLM error on study=s52026137 qtype=open: RetryError[<Future at 0x7af4e22bd820 state=finished raised AttributeError>]
21:24:29 | WARNING | cxr_intel.qa_dataset.synth_generator | LLM error on study=s52026137 qtype=severity: RetryError[<Future at 0x7af4e1bc3950 state=finished raised AttributeError>]
21:24:41 | WARNING | cxr_intel.qa_dataset.synth_generator | LLM error on study=s52026137 qtype=location: RetryError[<Future at 0x7af4e22bd820 state=finished raised AttributeError>]
synth->qa_v1.jsonl:  15% 3/20 [01:11<05:09, 18.18s/it]21:25:15 | WARNING | cxr_intel.qa_dataset.synth_gene

In [23]:
%cd /content/cxr-intel
!pip install -e . 2>&1 | tail -5

# In case pip install didn't take, add src/ to path directly
import sys
if '/content/cxr-intel/src' not in sys.path:
    sys.path.insert(0, '/content/cxr-intel/src')

# Verify
import cxr_intel
print("cxr_intel loaded from:", cxr_intel.__file__)


/content/cxr-intel
  Attempting uninstall: cxr-intel
    Found existing installation: cxr-intel 0.1.0
    Uninstalling cxr-intel-0.1.0:
      Successfully uninstalled cxr-intel-0.1.0
cxr_intel loaded from: /content/cxr-intel/src/cxr_intel/__init__.py


In [24]:
from google.colab import userdata
import os
os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_API_KEY')
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['QA_SYNTH_MODEL'] = 'meta/llama-3.3-70b-instruct'

from cxr_intel.models.llm_router import LLMRouter
r = LLMRouter(model='meta/llama-3.3-70b-instruct')
print(r.chat("You are helpful.", "Say 'NVIDIA NIM is working' in 5 words."))


22:34:03 | INFO    | cxr_intel.models.llm_router | LLMRouter using provider=nvidia model=meta/llama-3.3-70b-instruct
NVIDIA NIM is now working.


In [25]:
!python scripts/build_qa_dataset.py --config configs/data.yaml --limit 20


synth->qa_v1.jsonl:   0% 0/20 [00:00<?, ?it/s]22:34:55 | INFO    | cxr_intel.models.llm_router | LLMRouter using provider=nvidia model=meta/llama-3.3-70b-instruct
synth->qa_v1.jsonl:  50% 10/20 [05:50<02:50, 17.10s/it]22:41:21 | WARNING | cxr_intel.qa_dataset.synth_generator | LLM error on study=s51122130 qtype=open: RetryError[<Future at 0x7bb1c8dfef60 state=finished raised RateLimitError>]
synth->qa_v1.jsonl:  55% 11/20 [06:28<03:31, 23.51s/it]22:41:37 | WARNING | cxr_intel.qa_dataset.synth_generator | LLM error on study=s51106886 qtype=severity: RetryError[<Future at 0x7bb1c8dfe1e0 state=finished raised RateLimitError>]
22:41:50 | WARNING | cxr_intel.qa_dataset.synth_generator | LLM error on study=s51106886 qtype=location: RetryError[<Future at 0x7bb1c8c45250 state=finished raised RateLimitError>]
22:42:06 | WARNING | cxr_intel.qa_dataset.synth_generator | LLM error on study=s51106886 qtype=existence: RetryError[<Future at 0x7bb1c8ee71d0 state=finished raised RateLimitError>]
synth-

In [20]:
%cd /content/cxr-intel
!pip install -q -e .


/content/cxr-intel
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cxr-intel (pyproject.toml) ... done


In [26]:
from cxr_intel.utils.io import load_jsonl
qa = list(load_jsonl('data/qa/qa_v1.jsonl'))
print(f"Generated {len(qa)} validated QA pairs\n")
for q in qa[:5]:
    print(f"[{q['question_type']} | anchor={q['anchor_label']} | val={q['anchor_value']}]")
    print(f"  Q: {q['question']}")
    print(f"  A: {q['answer']}")
    print(f"  source: {q['source_sentence'][:140]}")
    print(f"  quality: {q['quality_scores']}")
    print()


Generated 45 validated QA pairs

[existence | anchor=Atelectasis | val=1.0]
  Q: Is there evidence of atelectasis?
  A: Yes
  source: The visualized lung portions show basal atelectasis, likely secondary to the herniated viscera.
  quality: {'correctness': 4.0, 'consistency': 4.0, 'completeness': 4.0, 'clinical_relevance': 4.0}

[open | anchor=Atelectasis | val=1.0]
  Q: What is identified on the radiograph in relation to the lung?
  A: Basal atelectasis
  source: The visualized lung portions show basal atelectasis, likely secondary to the herniated viscera.
  quality: {'correctness': 4.0, 'consistency': 4.0, 'completeness': 4.0, 'clinical_relevance': 4.0}

[severity | anchor=Atelectasis | val=1.0]
  Q: What is the extent of the atelectasis?
  A: Basal atelectasis is seen
  source: The visualized lung portions show basal atelectasis, likely secondary to the herniated viscera.
  quality: {'correctness': 4.0, 'consistency': 4.0, 'completeness': 4.0, 'clinical_relevance': 4.0}

[location 

In [27]:
%cd /content/cxr-intel
!git pull
!pip install -q -e .


/content/cxr-intel
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.92 KiB | 1.92 MiB/s, done.
From https://github.com/jo3591/assigment2dsai413
   b592fdd..12b3bbf  main       -> origin/main
Updating b592fdd..12b3bbf
Fast-forward
 src/cxr_intel/qa_dataset/synth_generator.py | 104 +++++++++++++++++-----------
 1 file changed, 63 insertions(+), 41 deletions(-)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cxr-intel (pyproject.toml) ... done


In [28]:
from google.colab import userdata
import os
for k in ['HF_TOKEN','OPENROUTER_API_KEY','KAGGLE_USERNAME','KAGGLE_KEY','NVIDIA_API_KEY','QA_SYNTH_MODEL','QA_JUDGE_MODEL']:
    try: os.environ[k] = userdata.get(k)
    except: pass
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['QA_SYNTH_MODEL'] = 'meta/llama-3.1-8b-instruct'
os.environ['QA_JUDGE_MODEL'] = 'meta/llama-3.3-70b-instruct'
print({k: os.environ.get(k, '<not set>')[:20] for k in ['LLM_PROVIDER','QA_SYNTH_MODEL']})


{'LLM_PROVIDER': 'nvidia', 'QA_SYNTH_MODEL': 'meta/llama-3.1-8b-in'}


In [29]:
!python scripts/build_qa_dataset.py --config configs/data.yaml --limit 10


synth->qa_v1.jsonl:   0% 0/10 [00:00<?, ?it/s]00:00:29 | INFO    | cxr_intel.models.llm_router | LLMRouter using provider=nvidia model=meta/llama-3.1-8b-instruct
00:00:29 | INFO    | cxr_intel.models.llm_router | LLMRouter using provider=nvidia model=meta/llama-3.1-8b-instruct
00:00:29 | INFO    | cxr_intel.models.llm_router | LLMRouter using provider=nvidia model=meta/llama-3.1-8b-instruct
00:00:29 | INFO    | cxr_intel.models.llm_router | LLMRouter using provider=nvidia model=meta/llama-3.1-8b-instruct
synth->qa_v1.jsonl: 100% 10/10 [00:18<00:00,  1.82s/it]
/content/cxr-intel/scripts/build_qa_dataset.py:89: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  n = save_jsonl((p.dict() for p in all_pairs), out_path)
00:00:46 | INFO    | build_qa_dataset | Wrote 34 QA pairs -> data/qa/qa_v1.jsonl
synth->qa_test.jsonl:  20% 2

In [ ]:
# 1. Re-mount Drive + secrets
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
for k in ['HF_TOKEN','OPENROUTER_API_KEY','KAGGLE_USERNAME','KAGGLE_KEY','NVIDIA_API_KEY']:
    try: os.environ[k] = userdata.get(k)
    except: pass
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['QA_SYNTH_MODEL'] = 'meta/llama-3.1-8b-instruct'
os.environ['QA_JUDGE_MODEL'] = 'meta/llama-3.3-70b-instruct'

# 2. Re-clone repo + install
!git clone https://github.com/jo3591/assigment2dsai413 /content/cxr-intel 2>/dev/null
%cd /content/cxr-intel
!git pull
!pip install -q -r requirements-colab.txt
!pip install -q -e .

# 3. Check what's already in Drive
!ls -la /content/drive/MyDrive/cxr_intel_artifacts/ 2>/dev/null || echo "Drive backup not found"

